# [197. Rising Temperature](https://leetcode.com/problems/rising-temperature/description/?envType=study-plan-v2&envId=top-sql-50)

In [1]:
import os

!pip install -q pyspark==3.4.1 #delta-spark==2.4.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.


In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()

## Example 1

### Input

**Weather**

| id | recordDate | temperature |
|----|------------|-------------|
| 1 | 2015-01-01 | 10 |
| 2 | 2015-01-02 | 25 |
| 3 | 2015-01-03 | 20 |
| 4 | 2015-01-04 | 30 |

### Output

| id |
|----|
| 2 |
| 4 |

### Explanation

- On `2015-01-02`, the temperature increased from `10` to `25`.
- On `2015-01-04`, the temperature increased from `20` to `30`.


## Example 2

### Input

**Weather**

| id | recordDate | temperature |
|----|------------|-------------|
| 1 | 2000-12-14 | 3 |
| 2 | 2000-12-16 | 5 |

### Output

| id |
|----|
|    |

### Explanation

There is no record for `2000-12-15`.

Although the temperature on `2000-12-16` is higher than on `2000-12-14`, the problem requires comparing with **yesterday's date**, not the previous available record.

Therefore, no rows are returned.

In [3]:
weather_data = [
    (1, "2015-01-01", 10),
    (2, "2015-01-02", 25),
    (3, "2015-01-03", 20),
    (4, "2015-01-04", 30)
]

weather_df = spark.createDataFrame(
    weather_data,
    ["id", "recordDate", "temperature"]
)

weather_df.show()

+---+----------+-----------+
| id|recordDate|temperature|
+---+----------+-----------+
|  1|2015-01-01|         10|
|  2|2015-01-02|         25|
|  3|2015-01-03|         20|
|  4|2015-01-04|         30|
+---+----------+-----------+



In [14]:
weather_data2 = [
    (1, "2000-12-14", 3),
    (2, "2000-12-16", 5)
]

weather_df2 = spark.createDataFrame(
    weather_data2,
    ["id", "recordDate", "temperature"]
)

weather_df2.show()

+---+----------+-----------+
| id|recordDate|temperature|
+---+----------+-----------+
|  1|2000-12-14|          3|
|  2|2000-12-16|          5|
+---+----------+-----------+



In [6]:
from pyspark.sql import Window
from pyspark.sql.functions import *

In [11]:
weather_df.withColumn(
      "old_temp",lag(col("temperature"),1).over(Window.orderBy(col("recordDate")))
    )\
    .withColumn(
        "old_date",lag(col("recordDate"),1).over(Window.orderBy(col("recordDate")))
    )\
    .filter(
        (col("temperature")>col("old_temp")) &
        (datediff(col("recordDate"),col("old_date"))==1)
    )\
.show()

+---+----------+-----------+--------+----------+
| id|recordDate|temperature|old_temp|  old_date|
+---+----------+-----------+--------+----------+
|  2|2015-01-02|         25|      10|2015-01-01|
|  4|2015-01-04|         30|      20|2015-01-03|
+---+----------+-----------+--------+----------+



In [15]:
weather_df2.withColumn(
      "old_temp",lag(col("temperature"),1).over(Window.orderBy(col("recordDate")))
    )\
    .withColumn(
        "old_date",lag(col("recordDate"),1).over(Window.orderBy(col("recordDate")))
    )\
    .filter(
        (col("temperature")>col("old_temp")) &
        (datediff(col("recordDate"),col("old_date"))==1)
    )\
.show()

+---+----------+-----------+--------+--------+
| id|recordDate|temperature|old_temp|old_date|
+---+----------+-----------+--------+--------+
+---+----------+-----------+--------+--------+

